In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

In [2]:
df = pd.read_csv("Financialdata_Segmented.csv")

In [4]:
median = df["Financial_Health_Score"].median()
df["Performance"] = np.where(
    df["Financial_Health_Score"] >= median,
    1,
    0
)

In [5]:
features = [

    "Gross_Margin",
    "Operating_Margin",
    "Net_Profit_Margin",
    "ROA",
    "ROE",
    "Debt_to_Assets",
    "Debt_to_Equity",
    "Asset_Turnover"

]
features = [c for c in features if c in df.columns]
X = df[features]
y = df["Performance"]

In [6]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [7]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

In [8]:
results = []
for name, model in models.items():
    model.fit(X_train,y_train)
    pred = model.predict(X_test)
    results.append({
        "Model":name,
        "Accuracy":accuracy_score(y_test,pred),
        "Precision":precision_score(y_test,pred),
        "Recall":recall_score(y_test,pred),
        "F1 Score":f1_score(y_test,pred)
    })
comparison = pd.DataFrame(results)
print(comparison)

                 Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression  0.973753   0.952153  1.000000  0.975490
1        Decision Tree  0.975066   0.974937  0.977387  0.976161
2        Random Forest  0.984252   0.984925  0.984925  0.984925


In [10]:
params = {
    "n_estimators":[100,200,300],
    "max_depth":[5,10,20,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]
}
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
grid.fit(X_train,y_train)
print("Best Parameters")
print(grid.best_params_)
print()
print("Best Accuracy")
print(grid.best_score_)

Best Parameters
{'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}

Best Accuracy
0.9796505962475438


In [11]:
best_classifier = grid.best_estimator_
pred = best_classifier.predict(X_test)
print("Final Accuracy")
print(accuracy_score(y_test,pred))

Final Accuracy
0.9803149606299213


In [15]:
target = '2015_PRICE_VAR_[Percent]'
X = df[features]
y = df[target]

In [16]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [17]:
regressors = {
    "Linear Regression":LinearRegression(),
    "Decision Tree":DecisionTreeRegressor(random_state=42),
    "Random Forest":RandomForestRegressor(random_state=42)
}

In [18]:
results=[]
for name,model in regressors.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    results.append({
        "Model":name,
        "MAE":mean_absolute_error(y_test,pred),
        "RMSE":np.sqrt(mean_squared_error(y_test,pred)),
        "R2":r2_score(y_test,pred)
    })
comparison=pd.DataFrame(results)
print(comparison)

               Model        MAE       RMSE        R2
0  Linear Regression  25.511416  33.045692  0.017528
1      Decision Tree  34.810937  45.591245 -0.870050
2      Random Forest  25.911418  34.255235 -0.055709


In [19]:
params={
    "n_estimators":[100,200,300],
    "max_depth":[10,20,None],
    "min_samples_split":[2,5],
    "min_samples_leaf":[1,2]
}
grid=GridSearchCV(
    RandomForestRegressor(random_state=42),
    params,
    cv=5,
    scoring="r2",
    n_jobs=-1
)
grid.fit(X_train,y_train)
print(grid.best_params_)
print(grid.best_score_)

{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
0.025426734586295983


In [20]:
best_model=grid.best_estimator_
pred=best_model.predict(X_test)
print("MAE")
print(mean_absolute_error(y_test,pred))
print()
print("RMSE")
print(np.sqrt(mean_squared_error(y_test,pred)))
print()
print("R²")
print(r2_score(y_test,pred))

MAE
25.214475009422973

RMSE
33.38119987224182

R²
-0.0025227211762486323


In [21]:
classification_results = comparison
classification_results.to_csv(
    "Classification_Model_Comparison.csv",
    index=False
)

In [22]:
regression_results = comparison
regression_results.to_csv(
    "Regression_Model_Comparison.csv",
    index=False
)